<a href="https://colab.research.google.com/github/gabrielhierro/LinguagensDeProgramacao/blob/main/AtividadePraticaSQLAlchemy/Atividade_Pratica_SQL_Alchemy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



### Nível 1: Básico



**Configuração e SQL Puro com Segurança**

Nesta etapa, os alunos farão a conexão inicial e utilizarão SQL nativo, compreendendo a importância de proteger o banco de dados contra ataques.

**Passo 1:** Crie a conexão com um banco de dados local utilizando a função create_engine('sqlite:///sistema_rh.db').


In [8]:
from sqlalchemy import create_engine

# Criar a conexão com o banco de dados local SQLite
engine = create_engine('sqlite:///sistema_rh.db', echo=True)

print("Conexão criada com sucesso!")

Conexão criada com sucesso!


**Passo 2:** Abra uma transação com with engine.begin() as conn: e utilize a função text() para executar um comando CREATE TABLE em SQL puro, criando uma tabela chamada funcionarios (com id, nome, cargo e salario).


In [9]:
from sqlalchemy import text

# Abrindo uma transação com o banco de dados
with engine.begin() as conn:
    # Executando o comando SQL puro para criar a tabela 'funcionarios'
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
    """))

print("Tabela 'funcionarios' criada com sucesso!")

2026-09-23 15:25:39,600 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:25:39,609 INFO sqlalchemy.engine.Engine 
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
    


INFO:sqlalchemy.engine.Engine:
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
    


2026-09-23 15:25:39,618 INFO sqlalchemy.engine.Engine [generated in 0.00888s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00888s] ()


2026-09-23 15:25:39,625 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Tabela 'funcionarios' criada com sucesso!


**Passo 3 (Segurança):** Simule a inserção de um novo funcionário a partir de um formulário web. Utilize o comando INSERT passando parâmetros seguros (ex: :nome, :cargo) e um dicionário de valores correspondentes. Pergunta reflexiva para os alunos: Por que nunca devemos concatenar strings diretamente no SQL (risco de injeção SQL) e como os placeholders resolvem isso?


In [10]:
# Simulação de dados inseridos num formulário web
dados_formulario = {
    "nome": "João Silva",
    "cargo": "Desenvolvedor Júnior",
    "salario": 3500.00
}

# Abrindo a transação
with engine.begin() as conn:
    # Utilizando placeholders (:nome, :cargo, :salario) para garantir a segurança
    conn.execute(
        text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)"),
        dados_formulario
    )

print("Funcionário inserido com segurança!")

# --- Reflexão ---
# Nunca devemos concatenar strings diretamente no SQL porque isso abre brechas para SQL Injection,
# permitindo que um utilizador mal-intencionado injete e execute comandos SQL indesejados (como um DROP TABLE).
# Os placeholders resolvem este problema porque a biblioteca (ou o driver da base de dados) trata a entrada
# estritamente como dados/valores, escapando caracteres perigosos antes da execução.

2026-09-23 15:27:22,039 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:27:22,042 INFO sqlalchemy.engine.Engine INSERT INTO funcionarios (nome, cargo, salario) VALUES (?, ?, ?)


INFO:sqlalchemy.engine.Engine:INSERT INTO funcionarios (nome, cargo, salario) VALUES (?, ?, ?)


2026-09-23 15:27:22,044 INFO sqlalchemy.engine.Engine [generated in 0.00204s] ('João Silva', 'Desenvolvedor Júnior', 3500.0)


INFO:sqlalchemy.engine.Engine:[generated in 0.00204s] ('João Silva', 'Desenvolvedor Júnior', 3500.0)


2026-09-23 15:27:22,048 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Funcionário inserido com segurança!


**Passo 4:** Valide a inserção consultando os dados com a função pd.read_sql_query(), que já retorna a tabela formatada diretamente como um DataFrame do Pandas.


In [11]:
import pandas as pd

# Abrindo a conexão e consultando os dados com o Pandas
with engine.connect() as conn:
    # O Pandas executa a query e converte o resultado diretamente para um DataFrame
    df_funcionarios = pd.read_sql_query(text("SELECT * FROM funcionarios"), conn)

# Exibindo a tabela formatada
# No Google Colab, se usares apenas 'df_funcionarios' no final da célula, ele formata de forma ainda mais visual!
print(df_funcionarios)

2026-09-23 15:28:27,717 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:28:27,720 INFO sqlalchemy.engine.Engine SELECT * FROM funcionarios


INFO:sqlalchemy.engine.Engine:SELECT * FROM funcionarios


2026-09-23 15:28:27,721 INFO sqlalchemy.engine.Engine [generated in 0.00388s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00388s] ()


2026-09-23 15:28:27,726 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


   id        nome                 cargo  salario
0   1   Ana Costa  Desenvolvedor Júnior   3850.0
1   2  João Silva  Desenvolvedor Júnior   3500.0


### Nível 2: Intermediário



**SQLAlchemy Core (Automatização Programática)**

O objetivo agora é abandonar o SQL em texto e usar as estruturas Python do SQLAlchemy Core, ideais para scripts de manipulação de dados e relatórios.

**Passo 1:** Defina uma nova tabela chamada projetos de forma programática utilizando os objetos Table, MetaData e Column. Em seguida, crie a tabela fisicamente no banco com metadata.`create_all(engine)`.


In [12]:
from sqlalchemy import MetaData, Table, Column, Integer, String, Float

# Instanciando o MetaData, que atua como um catálogo das nossas tabelas
metadata = MetaData()

# Definindo a tabela 'projetos' de forma programática
# As colunas id, nome e orcamento são criadas como exemplo para a estrutura da tabela
projetos = Table(
    'projetos', metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome', String, nullable=False),
    Column('orcamento', Float)
)

# Criando a tabela fisicamente no banco de dados (neste caso, o SQLite)
metadata.create_all(engine)

print("Tabela 'projetos' criada com sucesso utilizando a estrutura programática!")

2026-09-23 15:31:32,411 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:31:32,415 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("projetos")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("projetos")


2026-09-23 15:31:32,417 INFO sqlalchemy.engine.Engine [raw sql] ()


INFO:sqlalchemy.engine.Engine:[raw sql] ()


2026-09-23 15:31:32,420 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Tabela 'projetos' criada com sucesso utilizando a estrutura programática!


**Passo 2:** Receba uma lista de dicionários contendo múltiplos projetos e faça uma inserção em lote `(bulk insert)` passando essa lista para o comando `conn.execute(insert(projetos), [lista_de_dicts])`.


In [13]:
from sqlalchemy import insert

# Lista de dicionários contendo múltiplos projetos
lista_projetos = [
    {"nome": "Website Institucional", "orcamento": 15000.00},
    {"nome": "App de Vendas", "orcamento": 45000.00},
    {"nome": "Migração de Servidores", "orcamento": 8500.00}
]

# Abrindo a transação
with engine.begin() as conn:
    # Inserção em lote (bulk insert) passando a lista de dicionários
    conn.execute(insert(projetos), lista_projetos)

print("Inserção em lote realizada com sucesso!")

2026-09-23 15:34:32,001 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:34:32,004 INFO sqlalchemy.engine.Engine INSERT INTO projetos (nome, orcamento) VALUES (?, ?)


INFO:sqlalchemy.engine.Engine:INSERT INTO projetos (nome, orcamento) VALUES (?, ?)


2026-09-23 15:34:32,006 INFO sqlalchemy.engine.Engine [generated in 0.00235s] [('Website Institucional', 15000.0), ('App de Vendas', 45000.0), ('Migração de Servidores', 8500.0)]


INFO:sqlalchemy.engine.Engine:[generated in 0.00235s] [('Website Institucional', 15000.0), ('App de Vendas', 45000.0), ('Migração de Servidores', 8500.0)]


2026-09-23 15:34:32,008 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Inserção em lote realizada com sucesso!




**Passo 3:** A diretoria aprovou um reajuste. Utilize a instrução `update(tabela).where(...).values(...)` para aumentar o salário apenas dos funcionários que ocupam o cargo de `'Desenvolvedor Júnior'`.



In [14]:
from sqlalchemy import update, Table

# Como a tabela 'funcionarios' foi criada com SQL puro no Nível 1,
# vamos refleti-la (carregar a sua estrutura da base de dados) para usarmos o SQLAlchemy Core
funcionarios = Table('funcionarios', metadata, autoload_with=engine)

# Abrindo a transação
with engine.begin() as conn:
    # Criando a instrução de update para aumentar o salário (exemplo: aumento de 15%)
    instrucao_update = (
        update(funcionarios)
        .where(funcionarios.c.cargo == 'Desenvolvedor Júnior')
        .values(salario=funcionarios.c.salario * 1.15)
    )

    # Executando a instrução
    conn.execute(instrucao_update)

print("Reajuste salarial aplicado com sucesso aos Desenvolvedores Juniores!")

2026-09-23 15:35:18,873 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:35:18,875 INFO sqlalchemy.engine.Engine PRAGMA main.table_xinfo("funcionarios")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_xinfo("funcionarios")


2026-09-23 15:35:18,876 INFO sqlalchemy.engine.Engine [raw sql] ()


...

2026-09-23 15:35:18,924 INFO sqlalchemy.engine.Engine [generated in 0.00219s] (1.15, 'Desenvolvedor Júnior')


INFO:sqlalchemy.engine.Engine:[generated in 0.00219s] (1.15, 'Desenvolvedor Júnior')


2026-09-23 15:35:18,925 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Reajuste salarial aplicado com sucesso aos Desenvolvedores Juniores!




**Passo 4:** Gere um relatório salarial agregando os dados. Construa um `select` combinando `func.avg()` para calcular a média salarial e `group_by()` para agrupar o resultado por `cargo`.


In [15]:
from sqlalchemy import select, func

# Abrindo a conexão com o banco de dados
with engine.connect() as conn:
    # Construindo a consulta programática com SQLAlchemy Core
    consulta_relatorio = (
        select(
            funcionarios.c.cargo,
            func.avg(funcionarios.c.salario).label('media_salarial')
        )
        .group_by(funcionarios.c.cargo)
    )

    # Executando a instrução
    resultado = conn.execute(consulta_relatorio)

    # Iterando sobre o resultado para exibir o relatório
    print("--- Relatório de Média Salarial por Cargo ---")
    for linha in resultado:
        # Formatando a saída para exibir duas casas decimais no salário
        print(f"Cargo: {linha.cargo} | Média Salarial: R$ {linha.media_salarial:.2f}")

2026-09-23 15:43:38,581 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:43:38,584 INFO sqlalchemy.engine.Engine SELECT funcionarios.cargo, avg(funcionarios.salario) AS media_salarial 
FROM funcionarios GROUP BY funcionarios.cargo


INFO:sqlalchemy.engine.Engine:SELECT funcionarios.cargo, avg(funcionarios.salario) AS media_salarial 
FROM funcionarios GROUP BY funcionarios.cargo


2026-09-23 15:43:38,586 INFO sqlalchemy.engine.Engine [generated in 0.00467s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00467s] ()


--- Relatório de Média Salarial por Cargo ---
Cargo: Desenvolvedor Júnior | Média Salarial: R$ 4226.25
2026-09-23 15:43:38,590 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


### Nível 3: Avançado — ORM (Orientação a Objetos e Relacionamentos)

Na fase final, a turma aplicará o padrão ORM (Object-Relational Mapping), que é amplamente utilizado no desenvolvimento de aplicações modernas.

**Passo 1:** Transforme as tabelas em classes Python. Utilize `declarative_base()` e defina as classes `Departamento` e `FuncionarioORM` mapeando as colunas com `mapped_column`.



In [16]:
from sqlalchemy.orm import declarative_base
from sqlalchemy.orm import Mapped, mapped_column
from sqlalchemy import Integer, String, Float, ForeignKey

# Instanciando a base declarativa
Base = declarative_base()

# Definindo a classe Departamento
class Departamento(Base):
    __tablename__ = 'departamentos'

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)

# Definindo a classe FuncionarioORM
class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)
    cargo: Mapped[str] = mapped_column(String)
    salario: Mapped[float] = mapped_column(Float, nullable=True)

    # Criando a relação de chave estrangeira com a tabela departamentos
    departamento_id: Mapped[int] = mapped_column(ForeignKey('departamentos.id'))

# Criando as tabelas fisicamente no banco de dados
Base.metadata.create_all(engine)

print("Classes Departamento e FuncionarioORM criadas e mapeadas com sucesso!")

2026-09-23 15:45:26,625 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:45:26,628 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("departamentos")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("departamentos")


2026-09-23 15:45:26,631 INFO sqlalchemy.engine.Engine [raw sql] ()


...

2026-09-23 15:45:26,662 INFO sqlalchemy.engine.Engine [no key 0.00118s] ()


INFO:sqlalchemy.engine.Engine:[no key 0.00118s] ()


2026-09-23 15:45:26,677 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Classes Departamento e FuncionarioORM criadas e mapeadas com sucesso!


**Passo 2:** Estabeleça a relação entre as classes configurando uma `ForeignKey` na tabela de funcionários e utilizando a função `relationship` em ambas as classes para permitir a navegação de objeto para objeto.


In [17]:
from sqlalchemy.orm import relationship

# Como estamos redefinindo as classes para adicionar os relacionamentos,
# utilizaremos o parâmetro __table_args__ = {'extend_existing': True}
# para o SQLAlchemy permitir a atualização no Colab sem erros.

class Departamento(Base):
    __tablename__ = 'departamentos'
    __table_args__ = {'extend_existing': True}

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)

    # Navegação: Um departamento possui vários funcionários
    funcionarios = relationship("FuncionarioORM", back_populates="departamento")

class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'
    __table_args__ = {'extend_existing': True}

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)
    cargo: Mapped[str] = mapped_column(String)
    salario: Mapped[float] = mapped_column(Float, nullable=True)

    # ForeignKey configurada
    departamento_id: Mapped[int] = mapped_column(ForeignKey('departamentos.id'))

    # Navegação: Um funcionário pertence a um departamento
    departamento = relationship("Departamento", back_populates="funcionarios")

# Garantindo que as alterações estejam refletidas no banco
Base.metadata.create_all(engine)

print("Relações bidirecionais configuradas com sucesso utilizando 'relationship'!")

2026-09-23 15:46:26,809 INFO sqlalchemy.engine.Engine BEGIN (implicit)


/tmp/ipykernel_1356/3468451333.py:7: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.Departamento, and will be replaced in the string-lookup table.
  class Departamento(Base):
/tmp/ipykernel_1356/3468451333.py:17: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.FuncionarioORM, and will be replaced in the string-lookup table.
  class FuncionarioORM(Base):
INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:46:26,812 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("departamentos")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("departamentos")


2026-09-23 15:46:26,816 INFO sqlalchemy.engine.Engine [raw sql] ()


...

2026-09-23 15:46:26,819 INFO sqlalchemy.engine.Engine [raw sql] ()


INFO:sqlalchemy.engine.Engine:[raw sql] ()


2026-09-23 15:46:26,820 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Relações bidirecionais configuradas com sucesso utilizando 'relationship'!


**Passo 3:** Crie uma fábrica de sessões com `sessionmaker(bind=engine)` e instancie uma `Session`. Crie um objeto da classe `Departamento` e adicione funcionários a ele. Persista todos no banco de dados de uma só vez utilizando `sessao.add()` e `sessao.commit()`.


In [19]:
from sqlalchemy.orm import declarative_base, relationship, sessionmaker
from sqlalchemy.orm import Mapped, mapped_column
from sqlalchemy import Integer, String, Float, ForeignKey

# Recriando a Base para limpar o cache de classes das células anteriores
Base = declarative_base()

# Redefinindo as classes com os relacionamentos de uma só vez
class Departamento(Base):
    __tablename__ = 'departamentos'
    __table_args__ = {'extend_existing': True}

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)

    funcionarios = relationship("FuncionarioORM", back_populates="departamento")

class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'
    __table_args__ = {'extend_existing': True}

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String, nullable=False)
    cargo: Mapped[str] = mapped_column(String)
    salario: Mapped[float] = mapped_column(Float, nullable=True)

    departamento_id: Mapped[int] = mapped_column(ForeignKey('departamentos.id'))

    departamento = relationship("Departamento", back_populates="funcionarios")

# Garantindo a criação no banco
Base.metadata.create_all(engine)

# ==========================================
# Execução do Passo 3
# ==========================================

# Criando a fábrica de sessões e instanciando a Sessão
Session = sessionmaker(bind=engine)
sessao = Session()

# Criando um objeto da classe Departamento
departamento_ti = Departamento(nome="Tecnologia da Informação")

# Criando objetos da classe FuncionarioORM
funcionario1 = FuncionarioORM(nome="Ana Silva", cargo="Desenvolvedora Backend", salario=7500.00)
funcionario2 = FuncionarioORM(nome="Carlos Souza", cargo="Cientista de Dados", salario=9200.00)

# Adicionando os funcionários ao departamento
departamento_ti.funcionarios.append(funcionario1)
departamento_ti.funcionarios.append(funcionario2)

# Persistindo todos de uma só vez
sessao.add(departamento_ti)
sessao.commit()

print("Departamento e funcionários criados e salvos no banco de dados com sucesso!")

# Fechando a sessão
sessao.close()

2026-09-23 15:53:56,872 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:53:56,874 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("departamentos")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("departamentos")


2026-09-23 15:53:56,875 INFO sqlalchemy.engine.Engine [raw sql] ()


...

2026-09-23 15:53:56,908 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2 (ordered; batch not supported)] ('Carlos Souza', 'Cientista de Dados', 9200.0, 1)


INFO:sqlalchemy.engine.Engine:[insertmanyvalues 2/2 (ordered; batch not supported)] ('Carlos Souza', 'Cientista de Dados', 9200.0, 1)


2026-09-23 15:53:56,911 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Departamento e funcionários criados e salvos no banco de dados com sucesso!


**Passo 4:** Faça uma consulta orientada a objetos para listar todos os funcionários do departamento de "TI". Utilize `sessao.execute(select(...)).scalars().all()` para que o SQLAlchemy retorne os objetos instanciados da classe em vez de linhas textuais. No final, utilize `sessao.close()` para fechar a sessão adequadamente.


In [20]:
from sqlalchemy import select

# Instanciando uma nova Sessão para realizar a consulta (pois fechamos no passo anterior)
sessao = Session()

# Construindo a consulta com select e join para filtrar pelo nome do departamento
stmt = (
    select(FuncionarioORM)
    .join(Departamento)
    .where(Departamento.nome == 'Tecnologia da Informação')
)

# Executando a consulta e obtendo os objetos instanciados
funcionarios_ti = sessao.execute(stmt).scalars().all()

print("Funcionários do Departamento de TI:")
# Como funcionarios_ti é uma lista de objetos FuncionarioORM, podemos acessar seus atributos usando a notação de ponto
for f in funcionarios_ti:
    print(f"- Nome: {f.nome} | Cargo: {f.cargo} | Salário: R$ {f.salario:.2f}")

# Fechando a sessão adequadamente, conforme solicitado
sessao.close()
print("\nSessão fechada com sucesso.")

2026-09-23 15:56:01,964 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-23 15:56:01,970 INFO sqlalchemy.engine.Engine SELECT funcionarios_orm.id, funcionarios_orm.nome, funcionarios_orm.cargo, funcionarios_orm.salario, funcionarios_orm.departamento_id 
FROM funcionarios_orm JOIN departamentos ON departamentos.id = funcionarios_orm.departamento_id 
WHERE departamentos.nome = ?


INFO:sqlalchemy.engine.Engine:SELECT funcionarios_orm.id, funcionarios_orm.nome, funcionarios_orm.cargo, funcionarios_orm.salario, funcionarios_orm.departamento_id 
FROM funcionarios_orm JOIN departamentos ON departamentos.id = funcionarios_orm.departamento_id 
WHERE departamentos.nome = ?


2026-09-23 15:56:01,972 INFO sqlalchemy.engine.Engine [generated in 0.00199s] ('Tecnologia da Informação',)


INFO:sqlalchemy.engine.Engine:[generated in 0.00199s] ('Tecnologia da Informação',)


Funcionários do Departamento de TI:
- Nome: Ana Silva | Cargo: Desenvolvedora Backend | Salário: R$ 7500.00
- Nome: Carlos Souza | Cargo: Cientista de Dados | Salário: R$ 9200.00
2026-09-23 15:56:01,975 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK



Sessão fechada com sucesso.
